In [1]:
from herbie import Herbie
import pandas as pd
from pathlib import Path

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │   /home/konurnordberg/.config/herbie/config.toml     │
 ╰──────────────────────────────────────────────────────╯



In [2]:
RAW_DIR = Path("../data/raw")

RDU_LAT = 35.88
RDU_LON = -78.79

In [3]:
H = Herbie(
    "2025-09-16 18:00",
    model="gfs",
    product="pgrb2.0p25",
    fxx=24,
)

H

✅ Found ┊ model=gfs ┊ product=pgrb2.0p25 ┊ 2025-Sep-16 18:00 UTC F24 ┊ GRIB2 @ aws ┊ IDX @ aws


▌▌Herbie GFS model pgrb2.0p25 product initialized 2025-Sep-16 18:00 UTC F24 ┊ source=aws

In [4]:
H.inventory(":TMP:2 m above")

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this
580,581,419872306,420390579.0,419872306-420390579,2025-09-16 18:00:00,2025-09-17 18:00:00,TMP,2 m above ground,24 hour fcst,:TMP:2 m above ground:24 hour fcst


In [5]:
ds = H.xarray(":TMP:2 m above")

ds

<xarray.Dataset> Size: 4MB
Dimensions:              (latitude: 721, longitude: 1440)
Coordinates:
    time                 datetime64[ns] 8B 2025-09-16T18:00:00
    step                 timedelta64[ns] 8B 1 days
    heightAboveGround    float64 8B 2.0
  * latitude             (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude            (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    valid_time           datetime64[ns] 8B 2025-09-17T18:00:00
    gribfile_projection  object 8B None
Data variables:
    t2m                  (latitude, longitude) float32 4MB 266.9 266.9 ... 234.1
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    model:                   gfs
    product:                 pgrb2.0p25
    description:             NOAA Global Forecast System (GFS)
    remote_grib:             https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.20...
    local_grib:              /home/konurnordberg/data/gfs/20250916/subset_f60...
    search:                  :TMP:2 m above

In [6]:
gfs_lon = RDU_LON

if float(ds.longitude.max()) > 180:
    gfs_lon = RDU_LON % 360

rdu = ds.sel(
    latitude=RDU_LAT,
    longitude=gfs_lon,
    method="nearest"
)

rdu

<xarray.Dataset> Size: 60B
Dimensions:              ()
Coordinates:
    time                 datetime64[ns] 8B 2025-09-16T18:00:00
    step                 timedelta64[ns] 8B 1 days
    heightAboveGround    float64 8B 2.0
    latitude             float64 8B 36.0
    longitude            float64 8B 281.2
    valid_time           datetime64[ns] 8B 2025-09-17T18:00:00
    gribfile_projection  object 8B None
Data variables:
    t2m                  float32 4B 293.2
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    model:                   gfs
    product:                 pgrb2.0p25
    description:             NOAA Global Forecast System (GFS)
    remote_grib:             https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.20...
    local_grib:              /home/konurnordberg/data/gfs/20250916/subset_f60...
    search:                  :TMP:2 m above

In [7]:
temp_k = float(rdu["t2m"].values)

temp_c = temp_k - 273.15
temp_f = temp_c * 9 / 5 + 32

print("GFS predicted temperature:", temp_f, "°F")

GFS predicted temperature: 68.16989257812504 °F


In [8]:
def get_gfs_temperature(init_time, forecast_hour):
    H = Herbie(
        init_time,
        model="gfs",
        product="pgrb2.0p25",
        fxx=forecast_hour,
        verbose=False
    )

    ds = H.xarray(
        ":TMP:2 m above",
        remove_grib=True
    )

    lon = RDU_LON

    if float(ds.longitude.max()) > 180:
        lon = RDU_LON % 360

    rdu = ds.sel(
        latitude=RDU_LAT,
        longitude=lon,
        method="nearest"
    )

    temp_k = float(rdu["t2m"].values)
    temp_f = (temp_k - 273.15) * 9 / 5 + 32

    valid_time = pd.Timestamp(rdu["valid_time"].values)

    return {
        "gfs_init_utc": pd.Timestamp(init_time),
        "forecast_hour": forecast_hour,
        "valid_time_utc": valid_time,
        "gfs_temp_f": temp_f,
        "gfs_grid_lat": float(rdu["latitude"].values),
        "gfs_grid_lon": float(rdu["longitude"].values)
    }

In [9]:
get_gfs_temperature(
    "2025-09-16 18:00",
    24
)

{'gfs_init_utc': Timestamp('2025-09-16 18:00:00'),
 'forecast_hour': 24,
 'valid_time_utc': Timestamp('2025-09-17 18:00:00'),
 'gfs_temp_f': 68.16989257812504,
 'gfs_grid_lat': 36.0,
 'gfs_grid_lon': 281.25}

In [10]:
FORECAST_HOURS = (
    list(range(10, 121))
    + list(range(123, 346, 3))
)

print("Number of forecast points:", len(FORECAST_HOURS))

Number of forecast points: 186


In [11]:
HISTORICAL_RUNS = [
    "2021-09-16 18:00",
    "2022-09-16 18:00",
    "2023-09-16 18:00",
    "2024-09-16 18:00",
    "2025-09-16 18:00"
]

In [12]:
historical_rows = []

for init_time in HISTORICAL_RUNS:
    print(f"\nProcessing {init_time}")

    for i, forecast_hour in enumerate(FORECAST_HOURS):

        print(
            f"\rForecast hour {forecast_hour} "
            f"({i + 1}/{len(FORECAST_HOURS)})",
            end=""
        )

        try:
            row = get_gfs_temperature(
                init_time,
                forecast_hour
            )

            historical_rows.append(row)

        except Exception as e:
            print(
                f"\nFailed: {init_time}, "
                f"F{forecast_hour}: {e}"
            )


Processing 2021-09-16 18:00
Forecast hour 345 (186/186)
Processing 2022-09-16 18:00
Forecast hour 345 (186/186)
Processing 2023-09-16 18:00
Forecast hour 345 (186/186)
Processing 2024-09-16 18:00
Forecast hour 345 (186/186)
Processing 2025-09-16 18:00
Forecast hour 345 (186/186)

In [13]:
historical_gfs = pd.DataFrame(historical_rows)

historical_gfs.head()

,gfs_init_utc,forecast_hour,valid_time_utc,gfs_temp_f,gfs_grid_lat,gfs_grid_lon
0,2021-09-16 18:00:00,10,2021-09-17 04:00:00,72.949989,36.0,281.25
1,2021-09-16 18:00:00,11,2021-09-17 05:00:00,71.714247,36.0,281.25
2,2021-09-16 18:00:00,12,2021-09-17 06:00:00,71.451069,36.0,281.25
3,2021-09-16 18:00:00,13,2021-09-17 07:00:00,69.781917,36.0,281.25
4,2021-09-16 18:00:00,14,2021-09-17 08:00:00,69.460841,36.0,281.25


In [14]:
print(historical_gfs.shape)

historical_gfs.head(10)

(930, 6)


,gfs_init_utc,forecast_hour,valid_time_utc,gfs_temp_f,gfs_grid_lat,gfs_grid_lon
0,2021-09-16 18:00:00,10,2021-09-17 04:00:00,72.949989,36.0,281.25
1,2021-09-16 18:00:00,11,2021-09-17 05:00:00,71.714247,36.0,281.25
2,2021-09-16 18:00:00,12,2021-09-17 06:00:00,71.451069,36.0,281.25
3,2021-09-16 18:00:00,13,2021-09-17 07:00:00,69.781917,36.0,281.25
4,2021-09-16 18:00:00,14,2021-09-17 08:00:00,69.460841,36.0,281.25
5,2021-09-16 18:00:00,15,2021-09-17 09:00:00,69.266932,36.0,281.25
6,2021-09-16 18:00:00,16,2021-09-17 10:00:00,69.070002,36.0,281.25
7,2021-09-16 18:00:00,17,2021-09-17 11:00:00,68.881971,36.0,281.25
8,2021-09-16 18:00:00,18,2021-09-17 12:00:00,71.119722,36.0,281.25
9,2021-09-16 18:00:00,19,2021-09-17 13:00:00,74.756855,36.0,281.25


In [15]:
historical_gfs.to_csv(
    RAW_DIR / "NOAA_GFS_FORECAST_HISTORICAL.csv",
    index=False
)

In [16]:
pd.read_csv(
    RAW_DIR / "NOAA_GFS_FORECAST_HISTORICAL.csv"
).head()

,gfs_init_utc,forecast_hour,valid_time_utc,gfs_temp_f,gfs_grid_lat,gfs_grid_lon
0,2021-09-16 18:00:00,10,2021-09-17 04:00:00,72.949989,36.0,281.25
1,2021-09-16 18:00:00,11,2021-09-17 05:00:00,71.714247,36.0,281.25
2,2021-09-16 18:00:00,12,2021-09-17 06:00:00,71.451069,36.0,281.25
3,2021-09-16 18:00:00,13,2021-09-17 07:00:00,69.781917,36.0,281.25
4,2021-09-16 18:00:00,14,2021-09-17 08:00:00,69.460841,36.0,281.25


In [17]:
FINAL_GFS_RUN = "2026-09-16 18:00"

future_rows = []

for i, forecast_hour in enumerate(FORECAST_HOURS):

    print(
        f"\rForecast hour {forecast_hour} "
        f"({i + 1}/{len(FORECAST_HOURS)})",
        end=""
    )

    try:
        row = get_gfs_temperature(
            FINAL_GFS_RUN,
            forecast_hour
        )

        future_rows.append(row)

    except Exception as e:
        print(
            f"\nFailed F{forecast_hour}: {e}"
        )

Forecast hour 345 (186/186)

In [18]:
future_gfs = pd.DataFrame(future_rows)

future_gfs.head()

,gfs_init_utc,forecast_hour,valid_time_utc,gfs_temp_f,gfs_grid_lat,gfs_grid_lon
0,2026-09-16 18:00:00,10,2026-09-17 04:00:00,69.389924,36.0,281.25
1,2026-09-16 18:00:00,11,2026-09-17 05:00:00,68.854451,36.0,281.25
2,2026-09-16 18:00:00,12,2026-09-17 06:00:00,67.350587,36.0,281.25
3,2026-09-16 18:00:00,13,2026-09-17 07:00:00,66.622139,36.0,281.25
4,2026-09-16 18:00:00,14,2026-09-17 08:00:00,65.846943,36.0,281.25


In [19]:
future_gfs.to_csv(
    RAW_DIR / "NOAA_GFS_FORECAST_FUTURE.csv",
    index=False
)